In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    brier_score_loss,
    roc_curve
)

In [2]:
df = pd.read_csv("../data/raw/cs-training.csv")

df = df.drop(columns=["Unnamed: 0"])
df = df[df["age"] > 0]

df = df[
    ~df["NumberOfTimes90DaysLate"].isin([96, 98])
]

print(df.shape)
print(df["SeriousDlqin2yrs"].mean())

(149730, 11)
0.06597876177118814


In [3]:
X = df.drop(columns=["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

# First separate final test set
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.15,
    random_state=42,
    stratify=y
)

# Validation must represent 15% of the original dataset
validation_size = 0.15 / 0.85

X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=validation_size,
    random_state=42,
    stratify=y_temp
)

In [4]:
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nDefault rates:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())

Train: (104810, 10)
Validation: (22460, 10)
Test: (22460, 10)

Default rates:
Train: 0.06597652895716058
Validation: 0.0659839715048976
Test: 0.0659839715048976


In [5]:
income_median = X_train["MonthlyIncome"].median()
dependents_median = X_train["NumberOfDependents"].median()

print("Training income median:", income_median)
print("Training dependents median:", dependents_median)

Training income median: 5400.0
Training dependents median: 0.0


In [6]:
for dataset in [X_train, X_val, X_test]:
    dataset["MonthlyIncome"] = (
        dataset["MonthlyIncome"]
        .fillna(income_median)
    )

    dataset["NumberOfDependents"] = (
        dataset["NumberOfDependents"]
        .fillna(dependents_median)
    )

In [7]:
print("Train missing:", X_train.isnull().sum().sum())
print("Validation missing:", X_val.isnull().sum().sum())
print("Test missing:", X_test.isnull().sum().sum())

Train missing: 0
Validation missing: 0
Test missing: 0


## Final Model Validation Framework

To obtain an unbiased final performance estimate, the dataset was divided into three independent samples:

- 70% Training: used to learn WOE transformations and estimate the logistic regression.
- 15% Validation: used to select the classification threshold.
- 15% Test: reserved exclusively for final model evaluation.

All preprocessing parameters, including missing-value imputation and WOE binning, were learned using the training sample only.

In [8]:
continuous_features = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "DebtRatio",
    "MonthlyIncome"
]

delinquency_features = [
    "NumberOfTime30-59DaysPastDueNotWorse",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTimes90DaysLate"
]

In [9]:
def fit_continuous_woe(
    train_series,
    val_series,
    test_series,
    y_train,
    bins=10
):

    _, bin_edges = pd.qcut(
        train_series,
        q=bins,
        retbins=True,
        duplicates="drop"
    )

    bin_edges[0] = -np.inf
    bin_edges[-1] = np.inf

    train_bins = pd.cut(
        train_series,
        bins=bin_edges,
        include_lowest=True
    )

    val_bins = pd.cut(
        val_series,
        bins=bin_edges,
        include_lowest=True
    )

    test_bins = pd.cut(
        test_series,
        bins=bin_edges,
        include_lowest=True
    )

    temp = pd.DataFrame({
        "bin": train_bins,
        "target": y_train
    })

    table = (
        temp.groupby("bin", observed=False)["target"]
        .agg(["count", "sum"])
    )

    table["bad"] = table["sum"]
    table["good"] = table["count"] - table["bad"]

    table["dist_good"] = (
        table["good"] / table["good"].sum()
    )

    table["dist_bad"] = (
        table["bad"] / table["bad"].sum()
    )

    epsilon = 1e-10

    table["woe"] = np.log(
        (table["dist_good"] + epsilon) /
        (table["dist_bad"] + epsilon)
    )

    woe_map = table["woe"].to_dict()

    train_woe = train_bins.map(woe_map).astype(float)
    val_woe = val_bins.map(woe_map).astype(float)
    test_woe = test_bins.map(woe_map).astype(float)

    return train_woe, val_woe, test_woe, table

In [10]:
def delinquency_group(series):

    return pd.cut(
        series,
        bins=[-1, 0, 1, 2, 3, np.inf],
        labels=["0", "1", "2", "3", "4+"]
    )

In [11]:
def fit_grouped_woe(
    train_groups,
    val_groups,
    test_groups,
    y_train
):

    temp = pd.DataFrame({
        "group": train_groups,
        "target": y_train
    })

    table = (
        temp.groupby("group", observed=False)["target"]
        .agg(["count", "sum"])
    )

    table["bad"] = table["sum"]
    table["good"] = table["count"] - table["bad"]

    table["dist_good"] = (
        table["good"] / table["good"].sum()
    )

    table["dist_bad"] = (
        table["bad"] / table["bad"].sum()
    )

    epsilon = 1e-10

    table["woe"] = np.log(
        (table["dist_good"] + epsilon) /
        (table["dist_bad"] + epsilon)
    )

    woe_map = table["woe"].to_dict()

    train_woe = train_groups.map(woe_map).astype(float)
    val_woe = val_groups.map(woe_map).astype(float)
    test_woe = test_groups.map(woe_map).astype(float)

    return train_woe, val_woe, test_woe, table

In [12]:
X_train_woe = pd.DataFrame(index=X_train.index)
X_val_woe = pd.DataFrame(index=X_val.index)
X_test_woe = pd.DataFrame(index=X_test.index)

woe_tables = {}

In [13]:
for feature in continuous_features:

    train_w, val_w, test_w, table = fit_continuous_woe(
        X_train[feature],
        X_val[feature],
        X_test[feature],
        y_train,
        bins=10
    )

    X_train_woe[feature + "_woe"] = train_w
    X_val_woe[feature + "_woe"] = val_w
    X_test_woe[feature + "_woe"] = test_w

    woe_tables[feature] = table

In [14]:
for feature in delinquency_features:

    train_groups = delinquency_group(X_train[feature])
    val_groups = delinquency_group(X_val[feature])
    test_groups = delinquency_group(X_test[feature])

    train_w, val_w, test_w, table = fit_grouped_woe(
        train_groups,
        val_groups,
        test_groups,
        y_train
    )

    X_train_woe[feature + "_woe"] = train_w
    X_val_woe[feature + "_woe"] = val_w
    X_test_woe[feature + "_woe"] = test_w

    woe_tables[feature] = table

In [15]:
print(X_train_woe.shape)
print(X_val_woe.shape)
print(X_test_woe.shape)

print("\nMissing:")
print("Train:", X_train_woe.isnull().sum().sum())
print("Validation:", X_val_woe.isnull().sum().sum())
print("Test:", X_test_woe.isnull().sum().sum())

(104810, 7)
(22460, 7)
(22460, 7)

Missing:
Train: 0
Validation: 0
Test: 0


In [16]:
final_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

final_model.fit(
    X_train_woe,
    y_train
)

,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",42
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in 

In [17]:
val_prob = final_model.predict_proba(
    X_val_woe
)[:, 1]

val_auc = roc_auc_score(
    y_val,
    val_prob
)

print("Validation ROC-AUC:", val_auc)

Validation ROC-AUC: 0.8588749681724277


In [18]:
threshold_results = []

for threshold in np.arange(0.05, 0.55, 0.01):

    val_pred = (
        val_prob >= threshold
    ).astype(int)

    threshold_results.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred),
        "recall": recall_score(y_val, val_pred),
        "f1": f1_score(y_val, val_pred)
    })

threshold_df = pd.DataFrame(threshold_results)

best_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

best_row

threshold    0.160000
precision    0.372719
recall       0.523617
f1           0.435466
Name: 11, dtype: float64

In [19]:
final_threshold = best_row["threshold"]

print("Selected threshold:", final_threshold)

Selected threshold: 0.16000000000000003


In [20]:
test_prob = final_model.predict_proba(
    X_test_woe
)[:, 1]

final_auc = roc_auc_score(
    y_test,
    test_prob
)

test_pred = (
    test_prob >= final_threshold
).astype(int)

print("FINAL TEST ROC-AUC:", final_auc)

print("\nFinal Threshold:", final_threshold)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

print(
    "\nBrier Score:",
    brier_score_loss(y_test, test_prob)
)

FINAL TEST ROC-AUC: 0.8461381334008548

Final Threshold: 0.16000000000000003

Confusion Matrix:
[[19791  1187]
 [  779   703]]

Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.94      0.95     20978
           1       0.37      0.47      0.42      1482

    accuracy                           0.91     22460
   macro avg       0.67      0.71      0.68     22460
weighted avg       0.92      0.91      0.92     22460


Brier Score: 0.05017414918023976


In [21]:
# Gini coefficient
gini = 2 * final_auc - 1

print("Gini:", gini)

Gini: 0.6922762668017095


In [22]:
fpr, tpr, roc_thresholds = roc_curve(
    y_test,
    test_prob
)

ks_values = tpr - fpr

ks = np.max(ks_values)
ks_index = np.argmax(ks_values)

ks_threshold = roc_thresholds[ks_index]

print("KS Statistic:", ks)
print("KS Threshold:", ks_threshold)

KS Statistic: 0.5329397200254389
KS Threshold: 0.05578904979166449


In [23]:
final_results = pd.DataFrame({
    "ActualDefault": y_test.values,
    "PD": test_prob
})

In [24]:
final_results["RiskBand"] = pd.cut(
    final_results["PD"],
    bins=[0, 0.05, 0.10, 0.20, 0.40, 1],
    labels=[
        "Very Low",
        "Low",
        "Moderate",
        "High",
        "Very High"
    ],
    include_lowest=True
)

In [25]:
final_risk_summary = (
    final_results
    .groupby("RiskBand", observed=False)
    .agg(
        borrowers=("PD", "count"),
        avg_pd=("PD", "mean"),
        actual_default_rate=("ActualDefault", "mean")
    )
)

final_risk_summary

,borrowers,avg_pd,actual_default_rate
RiskBand,,,
Very Low,15679,0.021263,0.019325
Low,3606,0.071725,0.083749
Moderate,1675,0.136450,0.151045
High,840,0.275146,0.320238
Very High,660,0.607967,0.537879


In [26]:
LGD_ASSUMPTION = 0.45
EAD_ASSUMPTION = 10000

final_results["LGD"] = LGD_ASSUMPTION
final_results["EAD"] = EAD_ASSUMPTION

final_results["ExpectedLoss"] = (
    final_results["PD"]
    * final_results["LGD"]
    * final_results["EAD"]
)

In [27]:
final_el_summary = (
    final_results
    .groupby("RiskBand", observed=False)
    ["ExpectedLoss"]
    .agg(["count", "mean", "sum"])
)

final_el_summary

,count,mean,sum
RiskBand,,,
Very Low,15679,95.682023,1.500198e+06
Low,3606,322.762899,1.163883e+06
Moderate,1675,614.024690,1.028491e+06
High,840,1238.158018,1.040053e+06
Very High,660,2735.849565,1.805661e+06


In [28]:
final_results.to_csv(
    "../data/processed/final_credit_risk_results.csv",
    index=False
)

In [29]:
final_results.head()

,ActualDefault,PD,RiskBand,LGD,EAD,ExpectedLoss
0,0,0.042467,Very Low,0.45,10000,191.099720
1,0,0.044836,Very Low,0.45,10000,201.760554
2,0,0.036785,Very Low,0.45,10000,165.534241
3,0,0.018944,Very Low,0.45,10000,85.246265
4,0,0.411452,Very High,0.45,10000,1851.533632


In [30]:
final_results.shape

(22460, 6)